## SQL Analysis — Bank Stress Monitor

Five analytical queries demonstrating SQL skills on real FDIC data.
Each query answers a specific risk management question.

| Query | Question | Concepts Used |
|-------|----------|--------------|
| 1 | Top 20 stressed banks | SELECT, ORDER BY, LIMIT |
| 2 | CRE concentration by state | GROUP BY, CASE WHEN |
| 3 | Stress distribution by tier | GROUP BY, correlated subquery |
| 4 | Default rates by year and tier | SUBSTR, GROUP BY multiple columns |
| 5 | Failed vs healthy scores | CASE WHEN, WHERE, AVG |

In [180]:
# ============================================================
# BANK STRESS MONITOR — Phase 4
# SQL Queries & Analysis
# ============================================================
# Purpose: Demonstrate SQL skills on real financial data
#          All queries run on SQLite database built from
#          FDIC public data
# ============================================================

import sqlite3
import pandas as pd
import os

# ── Rebuild database from saved CSVs ─────────────────────
print("Building database from saved CSVs...")

db_file = 'bank_stress.db'
if os.path.exists(db_file):
    os.remove(db_file)

conn = sqlite3.connect(db_file)

# Load and insert each table
tables = {
    'bank_scores': 'bank_stress_scores_2025Q4.csv',
    'watchlist_top100': 'bank_stress_watchlist_top100.csv',
    'state_summary': 'bank_stress_state_summary.csv',
    'tier_summary': 'bank_stress_tier_summary.csv',
    'model_weights': 'bank_stress_model_weights.csv',
    'validation_results': 'bank_stress_validation_results.csv',
    'historical_scores': 'bank_stress_backtest_sample.csv',
}

for table_name, csv_file in tables.items():
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        df.to_sql(
            table_name, conn,
            if_exists='replace', index=False
        )
        print(f"  ✅ {table_name}: {len(df):,} rows")
    else:
        print(f"  ❌ {csv_file} not found")

print(f"\nDatabase ready ✅")
print(f"\nAvailable tables:")
tables_list = pd.read_sql(
    "SELECT name FROM sqlite_master "
    "WHERE type='table'",
    conn
)
for t in tables_list['name']:
    n = pd.read_sql(
        f"SELECT COUNT(*) as n FROM {t}",
        conn
    ).iloc[0]['n']
    print(f"  {t:<35}: {n:,} rows")

Building database from saved CSVs...
  ✅ bank_scores: 4,303 rows
  ✅ watchlist_top100: 100 rows
  ✅ state_summary: 55 rows
  ✅ tier_summary: 4 rows
  ✅ model_weights: 10 rows
  ✅ validation_results: 6 rows
  ✅ historical_scores: 91,932 rows

Database ready ✅

Available tables:
  bank_scores                        : 4,303 rows
  watchlist_top100                   : 100 rows
  state_summary                      : 55 rows
  tier_summary                       : 4 rows
  model_weights                      : 10 rows
  validation_results                 : 6 rows
  historical_scores                  : 91,932 rows


In [182]:
# Show me the 20 most stressed banks, sorted by Historical Score highest first.

query_1 = """
    SELECT 
    name, 
    stname AS state, 
    size_tier, 
    ROUND(historical_score,2) AS hist_score, 
    ROUND(current_regime_score,2), stress_class, 
    ROUND(asset/1000,2) AS asset_millions
    
    FROM bank_scores
    ORDER BY historical_score DESC
    LIMIT 20
    
"""

result_1 = pd.read_sql(query_1,conn)

print(result_1.to_string(index=False))

                               NAME         state SIZE_TIER  hist_score  ROUND(current_regime_score,2)    STRESS_CLASS  asset_millions
                Citizens First Bank          Iowa Community       86.58                          84.12 DEEPLY STRESSED           359.0
                        Coulee Bank     Wisconsin Community       85.38                          79.20 DEEPLY STRESSED           543.0
                      Hometown Bank          Ohio Community       85.36                          78.58 DEEPLY STRESSED           247.0
                         Crown Bank     Minnesota Community       85.05                          81.57 DEEPLY STRESSED           444.0
            New Valley Bank & Trust Massachusetts Community       84.26                          78.99 DEEPLY STRESSED           338.0
                         Union Bank       Vermont  Mid-Size       84.07                          82.85 DEEPLY STRESSED          1616.0
               Monterey County Bank    California Commu

In [184]:
#Which states have the most banks exceeding the 300% CRE regulatory threshold? 
#Show total banks, how many are above threshold, what percentage, and the average CRE ratio.

query_2 = """

    SELECT STNAME, bank_count, high_cre_count,ROUND(high_cre_count*100.0/bank_count,1) AS high_cre_percent, ROUND(avg_cre_ratio,2) AS avg_cre_ratio



    FROM state_summary

    ORDER BY high_cre_count DESC
    LIMIT 10

    

"""

result_2 = pd.read_sql(query_2,conn)

print(result_2.to_string(index=False))

    STNAME  bank_count  high_cre_count  high_cre_percent  avg_cre_ratio
     Texas         350              84              24.0           2.06
California         120              57              47.5           2.76
   Georgia         129              45              34.9           2.31
  Illinois         332              40              12.0           1.49
  Oklahoma         172              40              23.3           2.02
  Missouri         198              40              20.2           1.88
 Wisconsin         156              39              25.0           2.13
 Tennessee         111              37              33.3           2.41
 Minnesota         228              37              16.2           1.73
   Florida          82              35              42.7           2.71


In [186]:
#For each size tier, show how many banks fall into each stress classification and what percentage of that tier they represent.

query_3 = """

    SELECT 
    size_tier, 
    stress_class, 
    COUNT(*) AS tier_count, 
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM bank_scores b2 WHERE b2.size_tier = b1.size_tier),2) AS tier_pct
    


    FROM bank_scores b1

    GROUP BY size_tier, stress_class
    ORDER BY size_tier,tier_count DESC

    

    

"""

result_3 = pd.read_sql(query_3,conn)

print(result_3.to_string(index=False))

     SIZE_TIER    STRESS_CLASS  tier_count  tier_pct
     Community          NORMAL        2420     74.19
     Community        ELEVATED         616     18.88
     Community     LEGACY RISK         105      3.22
     Community DEEPLY STRESSED         104      3.19
     Community   EMERGING RISK          17      0.52
         Large          NORMAL          26     83.87
         Large        ELEVATED           4     12.90
         Large     LEGACY RISK           1      3.23
Large Regional          NORMAL          86     70.49
Large Regional        ELEVATED          29     23.77
Large Regional     LEGACY RISK           3      2.46
Large Regional DEEPLY STRESSED           3      2.46
Large Regional   EMERGING RISK           1      0.82
      Mid-Size          NORMAL         667     75.11
      Mid-Size        ELEVATED         172     19.37
      Mid-Size     LEGACY RISK          23      2.59
      Mid-Size DEEPLY STRESSED          20      2.25
      Mid-Size   EMERGING RISK           6    

In [188]:
#What was the bank default rate each year from 2003-2023, broken down by size tier?
query_4 = """

    SELECT 
    
    SUBSTR(REPDTE,1,4) AS year,
    size_tier,
    COUNT(*) AS total_banks,
    SUM(`DEFAULT`) AS defaults,
    ROUND(SUM(`DEFAULT`)*100.0/COUNT(*),2) AS default_pct
    
    FROM historical_scores 
    GROUP BY year, size_tier
    ORDER BY year ASC
    

"""

result_4 = pd.read_sql(query_4,conn)

print(result_4.to_string(index=False))

year      SIZE_TIER  total_banks  defaults  default_pct
2003      Community         5257        20         0.38
2003          Large            8         0         0.00
2003 Large Regional           56         0         0.00
2003       Mid-Size          312         0         0.00
2004      Community         5169         1         0.02
2004          Large           12         0         0.00
2004 Large Regional           66         0         0.00
2004       Mid-Size          284         0         0.00
2005      Community         4963         5         0.10
2005          Large            4         0         0.00
2005 Large Regional           69         0         0.00
2005       Mid-Size          313         2         0.64
2006      Community         4906        41         0.84
2006          Large            9         3        33.33
2006 Large Regional           64         5         7.81
2006       Mid-Size          318        15         4.72
2007      Community         5113       322      

In [190]:
#The question: In the years leading up to and during the crisis (2005-2010), did banks that 
#eventually failed already show higher stress scores than healthy banks?

query_5 = """

    SELECT 
    
    SUBSTR(REPDTE,1,4) AS year,
    CASE WHEN `default` = 1 THEN 'failed' ELSE 'healthy' END AS bank_status,
    COUNT(*),
    ROUND(AVG(historical_score),1) AS avg_hist_score,
    ROUND(AVG(current_regime_score),1) AS avg_regime_score
    
    FROM historical_scores 
    WHERE SUBSTR(REPDTE, 1, 4) BETWEEN '2005' AND '2010'
    GROUP BY year,bank_status
    ORDER BY year,bank_status ASC
    

"""

result_5 = pd.read_sql(query_5,conn)

print(result_5.to_string(index=False))

year bank_status  COUNT(*)  avg_hist_score  avg_regime_score
2005      failed         7            49.2              47.2
2005     healthy      5342            49.8              49.9
2006      failed        64            57.4              55.5
2006     healthy      5233            49.7              49.7
2007      failed       461            62.8              60.1
2007     healthy      5194            49.8              49.8
2008      failed      1076            70.3              64.5
2008     healthy      4984            49.5              49.7
2009      failed      1131            75.0              66.3
2009     healthy      4767            48.7              49.1
2010      failed       726            76.1              66.1
2010     healthy      4608            49.3              49.5
